# STgram-MFN Colab Training Commands

Run these cells top to bottom in Google Colab with a GPU runtime. The notebook downloads the gearbox data from Drive, trains STgram-MFN, evaluates source and target test splits, and copies artifacts back to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/SARWAGYASHAH/Anomalous-Sound-Detection-using-Spectrograms.git'
BRANCH = 'stgram-mfn-colab'

%cd /content
!rm -rf project
!git clone --branch "$BRANCH" "$REPO_URL" project
%cd /content/project

In [ ]:
!pip install -q -r requirements-colab.txt
!pip install -q -e .

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from pathlib import Path

DRIVE_ZIP_URL = 'https://drive.google.com/file/d/1p6TDo1GpTWHQzfHRcxs7NgXtG4-xVzqS/view?usp=drive_link'
ZIP_PATH = Path('Data/dev_data_gearbox.zip')
DATA_DIR = Path('Data/gearbox')

ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ZIP_PATH.exists():
    !gdown --fuzzy "$DRIVE_ZIP_URL" -O Data/dev_data_gearbox.zip

if not DATA_DIR.exists():
    !unzip -q -o Data/dev_data_gearbox.zip -d Data/

!find Data/gearbox -maxdepth 2 -type d | sort
!find Data/gearbox/train -name '*.wav' | wc -l
!find Data/gearbox/source_test -name '*.wav' | wc -l
!find Data/gearbox/target_test -name '*.wav' | wc -l

In [ ]:
!python pipeline/05_train_stgram.py --config config/stgram_mfn.yaml

In [ ]:
!python pipeline/06_evaluate_stgram.py --config config/stgram_mfn.yaml --split source_test --assets-dir docs/assets
!python pipeline/06_evaluate_stgram.py --config config/stgram_mfn.yaml --split target_test --assets-dir docs/assets

In [ ]:
import json
from pathlib import Path

for split in ['source_test', 'target_test']:
    metrics_path = Path('artifacts/evaluation_stgram') / split / 'metrics.json'
    metrics = json.loads(metrics_path.read_text())
    print('\n', split)
    print(json.dumps(metrics['metrics'], indent=2))

In [ ]:
from IPython.display import Image, display
from pathlib import Path

paths = [
    Path('artifacts/models/stgram_mfn/latest.txt'),
    Path('artifacts/evaluation_stgram/source_test/roc_curve.png'),
    Path('artifacts/evaluation_stgram/source_test/anomaly_score_distribution.png'),
    Path('artifacts/evaluation_stgram/target_test/roc_curve.png'),
    Path('artifacts/evaluation_stgram/target_test/anomaly_score_distribution.png'),
]

latest_run = Path(paths[0].read_text().strip())
display(Image(filename=str(latest_run / 'training_loss.png')))
for path in paths[1:]:
    display(Image(filename=str(path)))

In [ ]:
!mkdir -p /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn
!cp -r artifacts/models/stgram_mfn /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn/models
!cp -r artifacts/evaluation_stgram /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn/evaluation
!cp -r docs/assets /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn/docs_assets
!find /content/drive/MyDrive/anomalous_sound_detection/stgram_mfn -maxdepth 3 -type f | sort | tail -50